# RadAssist — Phase 3 Preliminary EDA (VQA-RAD)

This notebook produces every table, figure and number in the Phase 3 doc. Run it top to bottom; the last cell prints each `[placeholder]` value for the doc, and figures are saved to `figures/`.

**What it does**
1. Loads the original OSF release of VQA-RAD (it carries clinician `question_type`, `answer_type`, `image_organ`, `phrase_type` and `qid_linked_id`, which the Hugging Face mirror drops).
2. Recovers the official train/test split and removes the same 4 duplicate triplets the Hugging Face mirror removes.
3. Runs the EDA: split and image coverage, question/answer types, yes/no balance, answer quality, missingness, train–test leakage, image inspection.
4. Writes the outputs later phases need: cleaned data, a validation split, an image-disjoint split, test leakage flags, an answer-variant map, and a polarity-labeling sheet.

**Data.** Download the release from <https://osf.io/89kps/> (`VQA_RAD Dataset Public.json` and `VQA_RAD Image Folder`) into `data/vqa_rad_osf/`, or set `DOWNLOAD = True` below. Any folder layout under `DATA_DIR` works; files are found by searching it.

In [1]:
# --- Setup -------------------------------------------------------------------
import os, re, json, math, random, warnings, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

try:
    from IPython.display import display
except ImportError:          # running as a plain script
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Where the OSF release lives. Override with the VQA_RAD_DIR environment variable.
DATA_DIR = Path(os.environ.get("VQA_RAD_DIR", "data/vqa_rad_osf"))
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("eda_outputs"); OUT_DIR.mkdir(exist_ok=True)

SUMMARY = {}   # every number the Phase 3 doc needs; printed in the last cell

# Plot style: recessive axes/grid, one fixed color per entity across all figures.
C_TRAIN, C_TEST = "#2a78d6", "#eb6834"
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "semibold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "font.size": 10,
})

def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print("DATA_DIR:", DATA_DIR.resolve())

pandas 3.0.2 | numpy 2.1.2 | matplotlib 3.10.8
DATA_DIR: I:\phase3\data\vqa_rad_osf


In [2]:
# --- Optional download (needs internet) ----------------------------------------
DOWNLOAD = False   # True → fetch the OSF release with osfclient into DATA_DIR
if DOWNLOAD and not list(DATA_DIR.rglob("*.json")):
    import subprocess, sys, zipfile
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osfclient"], check=True)
    subprocess.run(["osf", "-p", "89kps", "clone", str(DATA_DIR)], check=True)
    for z in DATA_DIR.rglob("*.zip"):          # the image folder may arrive zipped
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.parent)

## 1. Load the OSF release

In [3]:
json_files = sorted(DATA_DIR.rglob("*.json"), key=lambda p: p.stat().st_size, reverse=True)
json_files = [p for p in json_files if "vqa" in p.name.lower()] or json_files
assert json_files, f"No JSON found under {DATA_DIR}. Download the OSF release first (see the top of the notebook)."
JSON_PATH = json_files[0]
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)
if isinstance(raw, dict):                        # tolerate a wrapped list
    raw = next(v for v in raw.values() if isinstance(v, list))
df_raw = pd.DataFrame(raw)
print("Loaded", JSON_PATH.name, "->", df_raw.shape)
print("Columns:", list(df_raw.columns))

IMG_INDEX = {p.name: p for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
print("Image files found:", len(IMG_INDEX))

AssertionError: No JSON found under data\vqa_rad_osf. Download the OSF release first (see the top of the notebook).

In [ ]:
def clean_str(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return str(x).strip()

def norm_answer(a):
    """Lowercase, trim, drop punctuation except / - and decimal points, collapse spaces."""
    a = str(a).lower().strip()
    a = re.sub(r"[^\w\s/.-]", "", a)
    a = re.sub(r"(?<!\d)\.|\.(?!\d)", "", a)   # keep "5.6cm", drop sentence-final "."
    return re.sub(r"\s+", " ", a).strip()

df = df_raw.copy()
for col in ["qid", "image_name", "image_organ", "question", "answer", "answer_type",
            "question_type", "phrase_type", "qid_linked_id", "question_frame", "evaluation"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].map(clean_str)

df["image_organ"] = df["image_organ"].str.upper()
df["answer_type"] = df["answer_type"].str.upper()
df["question_type"] = df["question_type"].str.upper()
# Two question_type typos in the release: ATRIB -> ATTRIB, PRSE -> PRES
QTYPE_FIXES = {"ATRIB": "ATTRIB", "PRSE": "PRES"}
for bad, good in QTYPE_FIXES.items():
    df["question_type"] = df["question_type"].str.replace(rf"\b{bad}\b", good, regex=True)
df["phrase_type"] = df["phrase_type"].str.lower()
df["q_norm"] = df["question"].str.lower().str.strip()
df["a_norm"] = df["answer"].map(norm_answer)
# question_type can list several categories (e.g. "PRES, POS"); keep them as a list
df["qtypes"] = df["question_type"].map(lambda s: [t.strip() for t in re.split(r"[,/;]", s) if t.strip()])

for col in ["phrase_type", "answer_type", "image_organ", "evaluation"]:
    print(f"\n{col}:")
    print(df[col].replace("", "<empty>").value_counts().to_string())

## 2. Missingness

In [ ]:
miss = {
    "question_type": int((df["question_type"] == "").sum()),
    "answer_type": int((df["answer_type"] == "").sum()),
    "image_organ": int((df["image_organ"] == "").sum()),
    "phrase_type": int((df["phrase_type"] == "").sum()),
    "qid_linked_id": int((df["qid_linked_id"] == "").sum()),
    "image file not found": int((~df["image_name"].isin(IMG_INDEX.keys())).sum()),
}
missing_table = pd.DataFrame({"field": list(miss), "missing_or_empty": list(miss.values())})
display(missing_table)
missing_table.to_csv(OUT_DIR / "table_missingness.csv", index=False)
SUMMARY.update({
    "n_missing_question_type": miss["question_type"],
    "n_missing_answer_type": miss["answer_type"],
    "n_missing_image_organ": miss["image_organ"],
    "n_missing_image_file": miss["image file not found"],
})

## 3. Official split and duplicate removal

In [ ]:
# The authors' test set = 300 free-form questions + their 151 paraphrases; phrase_type marks them "test_*".
df["split"] = np.where(df["phrase_type"].str.startswith("test"), "test", "train")
print("Raw counts by split:", df["split"].value_counts().to_dict())
SUMMARY["n_raw"] = len(df)

key = ["image_name", "q_norm", "a_norm"]
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
dup_within_train = train.duplicated(subset=key, keep="first")
test_keys = set(map(tuple, test[key].values))
train_in_test = train[key].apply(tuple, axis=1).isin(test_keys)
drop_idx = train.index[dup_within_train | train_in_test]
print(f"Duplicates within train: {int(dup_within_train.sum())}; "
      f"train triplets also in test: {int(train_in_test.sum())}")
df = df.drop(index=drop_idx).reset_index(drop=True)
SUMMARY["n_dup_train"] = int(dup_within_train.sum())
SUMMARY["n_train_test_dup_removed"] = int(train_in_test.sum())

n_train, n_test = int((df.split == "train").sum()), int((df.split == "test").sum())
print(f"After cleaning: train {n_train}, test {n_test}, total {len(df)}")
if (n_train, n_test) != (1793, 451):
    print("WARNING: expected 1,793 / 451 (Hugging Face mirror). Check the phrase_type values above, "
          "or run the optional cross-check in the next cell.")
SUMMARY.update({"n_train": n_train, "n_test": n_test, "n_total_clean": len(df)})
df.drop(columns=["qtypes"]).to_csv(OUT_DIR / "vqa_rad_clean.csv", index=False)

In [ ]:
# Optional: cross-check the recovered split against the Hugging Face mirror (needs `datasets` + internet).
CHECK_HF = False
if CHECK_HF:
    from datasets import load_dataset
    hf = load_dataset("flaviagiammarino/vqa-rad")
    for split in ["train", "test"]:
        hf_pairs = Counter(zip([q.lower().strip() for q in hf[split]["question"]],
                               [norm_answer(a) for a in hf[split]["answer"]]))
        ours = Counter(zip(df.loc[df.split == split, "q_norm"], df.loc[df.split == split, "a_norm"]))
        matched = sum((hf_pairs & ours).values())
        print(f"{split}: HF {sum(hf_pairs.values())} | ours {sum(ours.values())} | matched pairs {matched}")

## 4. Split and image coverage

In [ ]:
img_train = set(df.loc[df.split == "train", "image_name"])
img_test = set(df.loc[df.split == "test", "image_name"])
shared = img_train & img_test
qpi = df.groupby("image_name").size()

split_table = pd.DataFrame({
    "": ["QA pairs", "Images"],
    "Train": [n_train, len(img_train)],
    "Test": [n_test, len(img_test)],
    "Total": [len(df), f"{len(img_train | img_test)} ({len(shared)} in both)"],
})
display(split_table)
split_table.to_csv(OUT_DIR / "table_split.csv", index=False)
print(f"Questions per image: median {qpi.median():.0f}, mean {qpi.mean():.1f}, max {qpi.max()}")
SUMMARY.update({
    "n_img_train": len(img_train), "n_img_test": len(img_test), "n_img_shared": len(shared),
    "q_per_image_median": float(qpi.median()), "q_per_image_max": int(qpi.max()),
})

## 5. Question and answer types (Figures E1, E2)

In [ ]:
qt = df[["split", "qtypes"]].explode("qtypes")
qt["qtypes"] = qt["qtypes"].fillna("<missing>")
qt_counts = qt.groupby(["qtypes", "split"]).size().unstack(fill_value=0)
for s in ["train", "test"]:
    if s not in qt_counts:
        qt_counts[s] = 0
qt_counts["total"] = qt_counts["train"] + qt_counts["test"]
qt_counts = qt_counts.sort_values("total", ascending=False)
print("Question-type counts (a question with two types counts once in each):")
display(qt_counts)
qt_counts.to_csv(OUT_DIR / "table_question_types.csv")

closed_mask = df["answer_type"] == "CLOSED"
print("Answer types:", df["answer_type"].value_counts().to_dict())
SUMMARY["pct_closed"] = round(100 * closed_mask.mean(), 1)
SUMMARY["top_qtype"] = qt_counts.index[0]
SUMMARY["n_top_qtype"] = int(qt_counts["total"].iloc[0])

# Figure E1: question types by split
order = list(qt_counts.index[::-1])
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(8, 0.38 * len(order) + 1.2))
ax.barh(y + 0.2, qt_counts.loc[order, "train"], height=0.4, color=C_TRAIN, edgecolor=SURFACE, linewidth=1.5, label="Train")
ax.barh(y - 0.2, qt_counts.loc[order, "test"], height=0.4, color=C_TEST, edgecolor=SURFACE, linewidth=1.5, label="Test")
ax.set_yticks(y, order)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Questions")
ax.set_title("Figure E1. Question types by split")
ax.legend(loc="lower right")
save(fig, "fig_e1_question_types.png")

In [ ]:
# Figure E2: organ x answer type, faceted by answer type, train vs test
organs = [o for o in ["HEAD", "CHEST", "ABD"] if o in set(df.image_organ)] + \
         sorted(set(df.image_organ) - {"HEAD", "CHEST", "ABD"})
atypes = [a for a in ["CLOSED", "OPEN"] if a in set(df.answer_type)] or sorted(set(df.answer_type))
ct = df.groupby(["answer_type", "image_organ", "split"]).size()
fig, axes = plt.subplots(1, len(atypes), figsize=(4.2 * len(atypes), 3.6), sharey=True, squeeze=False)
x = np.arange(len(organs))
for ax, at in zip(axes[0], atypes):
    tr = [ct.get((at, o, "train"), 0) for o in organs]
    te = [ct.get((at, o, "test"), 0) for o in organs]
    ax.bar(x - 0.2, tr, width=0.4, color=C_TRAIN, edgecolor=SURFACE, linewidth=1.5, label="Train")
    ax.bar(x + 0.2, te, width=0.4, color=C_TEST, edgecolor=SURFACE, linewidth=1.5, label="Test")
    ax.set_xticks(x, [o if o else "<missing>" for o in organs])
    ax.grid(axis="x", visible=False)
    ax.set_title(at.capitalize() if at else "<missing>")
axes[0][0].set_ylabel("Questions")
axes[0][-1].legend(loc="upper right")
fig.suptitle("Figure E2. Questions by organ and answer type", fontweight="semibold", color=INK)
fig.tight_layout()
save(fig, "fig_e2_organ_answer_type.png")
pd.crosstab([df.answer_type, df.image_organ], df.split).to_csv(OUT_DIR / "table_organ_answer_type.csv")

## 6. Class balance (yes/no)

In [ ]:
def yn_bucket(a):
    return a if a in ("yes", "no") else "other"

def is_presence(ts):
    return bool({"PRES", "ABN"} & set(ts))

closed = df[df.answer_type == "CLOSED"].copy()
closed["yn"] = closed["a_norm"].map(yn_bucket)
risk_candidates = closed[(closed.split == "test") & closed["qtypes"].map(is_presence)]
risk_subset = risk_candidates[risk_candidates.yn != "other"].copy()

rows = []
for name, d in [("Closed, train", closed[closed.split == "train"]),
                ("Closed, test", closed[closed.split == "test"]),
                ("Closed PRES/ABN, test (risk subset)", risk_subset)]:
    vc = d["yn"].value_counts(normalize=True) * 100
    rows.append({"subset": name, "n": len(d), "yes_%": round(vc.get("yes", 0.0), 1),
                 "no_%": round(vc.get("no", 0.0), 1), "other_%": round(vc.get("other", 0.0), 1)})
balance = pd.DataFrame(rows)
display(balance)
balance.to_csv(OUT_DIR / "table_class_balance.csv", index=False)

majority = closed.loc[closed.split == "train", "a_norm"].mode()[0]
maj_acc = 100 * (closed.loc[closed.split == "test", "a_norm"] == majority).mean()
print(f"Majority closed training answer: '{majority}' -> {maj_acc:.1f}% accuracy on closed test questions")
print(f"Closed PRES/ABN test questions with non-yes/no answers (outside the risk subset): "
      f"{len(risk_candidates) - len(risk_subset)}")

open_abn_test = df[(df.split == "test") & (df.answer_type == "OPEN") & df["qtypes"].map(lambda ts: "ABN" in ts)]
b = balance.set_index("subset")
SUMMARY.update({
    "closed_train_n": int(b.loc["Closed, train", "n"]),
    "closed_train_yes_pct": b.loc["Closed, train", "yes_%"],
    "closed_train_no_pct": b.loc["Closed, train", "no_%"],
    "closed_train_other_pct": b.loc["Closed, train", "other_%"],
    "closed_test_n": int(b.loc["Closed, test", "n"]),
    "closed_test_yes_pct": b.loc["Closed, test", "yes_%"],
    "closed_test_no_pct": b.loc["Closed, test", "no_%"],
    "closed_test_other_pct": b.loc["Closed, test", "other_%"],
    "presence_test_n": int(b.loc["Closed PRES/ABN, test (risk subset)", "n"]),
    "presence_test_yes_pct": b.loc["Closed PRES/ABN, test (risk subset)", "yes_%"],
    "presence_test_no_pct": b.loc["Closed PRES/ABN, test (risk subset)", "no_%"],
    "majority_closed_test_acc": round(maj_acc, 1),
    "n_test_abn_open": len(open_abn_test),
})

## 7. Answer quality and noise (Figure E3)

In [ ]:
# (a) Clinician-closed questions whose answer is not yes/no (limited-choice questions)
nonyesno_closed = closed[closed.yn == "other"]
print(f"Clinician-closed questions with non-yes/no answers: {len(nonyesno_closed)}")
display(nonyesno_closed[["question", "answer", "question_type", "split"]].head(10))
SUMMARY["n_closed_nonyesno"] = len(nonyesno_closed)

# (b) Spelling variants: answers that differ only by case, punctuation, hyphens or spacing
def alnum_key(a):
    return re.sub(r"[^a-z0-9]", "", str(a).lower())

groups = defaultdict(Counter)
for a in df["answer"]:
    if alnum_key(a):
        groups[alnum_key(a)][a.strip()] += 1
all_groups = {k: c for k, c in groups.items() if len(c) > 1}
# spelling variants: still different after lowercasing (hyphens, spacing, punctuation)
variant_groups = {k: c for k, c in all_groups.items() if len({v.lower() for v in c}) > 1}
case_only = len(all_groups) - len(variant_groups)
print(f"\nAnswer groups that differ only by case: {case_only}")
print(f"Spelling-variant groups (hyphens, spacing, punctuation): {len(variant_groups)} (largest shown)")
for k, c in sorted(variant_groups.items(), key=lambda kv: -sum(kv[1].values()))[:15]:
    print("  ", dict(c))
SUMMARY["n_variant_groups"] = len(variant_groups)
SUMMARY["n_case_only_groups"] = case_only

# Variant map for preprocessing: each normalized variant -> the most common normalized form
variant_map = {}
for c in variant_groups.values():
    normed = Counter()
    for v, n in c.items():
        normed[norm_answer(v)] += n
    canon = normed.most_common(1)[0][0]
    variant_map.update({v: canon for v in normed if v != canon})
with open(OUT_DIR / "answer_variant_map.json", "w") as f:
    json.dump(variant_map, f, indent=2, sort_keys=True)
print(f"Answer variant map: {len(variant_map)} entries -> {OUT_DIR / 'answer_variant_map.json'}")

In [ ]:
# (c) Long tail of open answers
open_ = df[df.answer_type == "OPEN"]
open_counts = open_["a_norm"].value_counts()
words = open_["a_norm"].str.split().str.len()
print(f"Open questions: {len(open_)} | distinct answers: {open_counts.size} | "
      f"seen once: {(open_counts == 1).sum()} | median words: {words.median():.0f}")
display(open_counts.head(20).rename("count").to_frame())
open_counts.to_csv(OUT_DIR / "table_open_answer_counts.csv", header=["count"])
SUMMARY.update({
    "n_open_unique": int(open_counts.size),
    "n_open_singleton": int((open_counts == 1).sum()),
    "open_answer_words_median": float(words.median()) if len(words) else 0.0,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ranks = np.arange(1, open_counts.size + 1)
ax1.plot(ranks, open_counts.values, color=C_TRAIN, linewidth=2)
ax1.set_xscale("log"); ax1.set_yscale("log")
for axis in (ax1.xaxis, ax1.yaxis):
    axis.set_major_locator(matplotlib.ticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    axis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax1.set_xlabel("Answer rank"); ax1.set_ylabel("Occurrences")
ax1.set_title("Open-answer frequency (log–log)")
maxw = int(words.max()) if len(words) else 1
ax2.hist(words, bins=np.arange(0.5, maxw + 1.5, 1), color=C_TRAIN, edgecolor=SURFACE, linewidth=1.5)
ax2.grid(axis="x", visible=False)
ax2.set_xlabel("Words in answer"); ax2.set_ylabel("Open questions")
ax2.set_title("Open-answer length")
fig.suptitle("Figure E3. Open answers: long tail and length", fontweight="semibold", color=INK)
fig.tight_layout()
save(fig, "fig_e3_open_answers.png")

## 8. Train–test leakage

Checks: (1) test images that also appear in training; (2) test questions whose linked paraphrase (`qid_linked_id`) is in training — the authors kept paraphrase pairs together, so this should be 0; (3) test questions with a near-identical training question on the same image (token Jaccard ≥ 0.6); (4) identical image–question–answer triplets in both splits after cleaning (should be 0). Per-question flags are saved for stratified reporting later.

In [ ]:
train_df = df[df.split == "train"]
test_df = df[df.split == "test"].copy()

link_sizes = df.loc[df.qid_linked_id != "", "qid_linked_id"].value_counts()
print(f"qid_linked_id: {link_sizes.size} distinct ids over {int(link_sizes.sum())} rows; "
      f"ids shared by >1 row: {int((link_sizes > 1).sum())}")
if link_sizes.size and int((link_sizes > 1).sum()) == 0:
    print("NOTE: every qid_linked_id is unique here, so it does not link paraphrases; inspect question_frame instead.")
train_links = set(train_df.loc[train_df.qid_linked_id != "", "qid_linked_id"])
test_df["para_in_train"] = (test_df.qid_linked_id != "") & test_df.qid_linked_id.isin(train_links)

def toks(q):
    return set(re.findall(r"[a-z0-9]+", q.lower()))

train_by_img = defaultdict(list)
for r in train_df.itertuples():
    train_by_img[r.image_name].append((toks(r.question), r.a_norm, r.question))

best_j, best_q, best_same = [], [], []
for r in test_df.itertuples():
    t, bj, bq, bs = toks(r.question), 0.0, "", False
    for tt, a, q in train_by_img.get(r.image_name, []):
        j = len(t & tt) / max(1, len(t | tt))
        if j > bj:
            bj, bq, bs = j, q, (a == r.a_norm)
    best_j.append(bj); best_q.append(bq); best_same.append(bs)
test_df["max_jaccard_same_image"] = best_j
test_df["closest_train_question"] = best_q
test_df["neardup"] = test_df["max_jaccard_same_image"] >= 0.6
test_df["neardup_same_answer"] = test_df["neardup"] & pd.Series(best_same, index=test_df.index)
test_df["image_in_train"] = test_df.image_name.isin(img_train)

exact_cross = set(map(tuple, train_df[key].values)) & set(map(tuple, test_df[key].values))

leak_table = pd.DataFrame({
    "check": ["Test images that also appear in training",
              "Test questions whose linked paraphrase is in training",
              "Test questions with a near-identical training question on the same image",
              "  ...of which the training answer is the same",
              "Identical image-question-answer triplets in both splits"],
    "result": [f"{len(shared)} of {len(img_test)}", int(test_df.para_in_train.sum()),
               int(test_df.neardup.sum()), int(test_df.neardup_same_answer.sum()), len(exact_cross)],
})
display(leak_table)
leak_table.to_csv(OUT_DIR / "table_leakage.csv", index=False)
print("\nExamples of near-duplicates:")
display(test_df.loc[test_df.neardup, ["image_name", "question", "closest_train_question", "answer"]].head(10))

test_df[["qid", "image_name", "question", "answer", "question_type", "answer_type", "image_in_train",
         "para_in_train", "max_jaccard_same_image", "closest_train_question", "neardup",
         "neardup_same_answer"]].to_csv(OUT_DIR / "test_leakage_flags.csv", index=False)
SUMMARY.update({
    "n_test_para_in_train": int(test_df.para_in_train.sum()),
    "n_test_neardup": int(test_df.neardup.sum()),
    "n_test_neardup_same_answer": int(test_df.neardup_same_answer.sum()),
    "n_exact_dup_cross": len(exact_cross),
})

## 9. Image inspection (Figures E4, E5)

In [ ]:
SINGLE_CHANNEL = {"1", "L", "LA", "I", "I;16", "I;16B", "I;16L", "F"}
rows = []
for name in sorted(set(df.image_name)):
    p = IMG_INDEX.get(name)
    if p is None:
        continue
    with Image.open(p) as im:
        gray_rgb = False
        if im.mode == "RGB":
            a = np.asarray(im.resize((64, 64))).astype(int)
            gray_rgb = bool((np.abs(a[..., 0] - a[..., 1]).max() <= 2) and (np.abs(a[..., 1] - a[..., 2]).max() <= 2))
        rows.append({"image_name": name, "width": im.width, "height": im.height,
                     "mode": im.mode, "rgb_but_gray": gray_rgb})
img_info = pd.DataFrame(rows)
organ_by_img = df.groupby("image_name")["image_organ"].agg(lambda s: s.mode().iat[0] if len(s.mode()) else "")
img_info["organ"] = img_info.image_name.map(organ_by_img)
img_info.to_csv(OUT_DIR / "image_info.csv", index=False)

print("Image modes:", img_info["mode"].value_counts().to_dict())
print(f"RGB files whose three channels are identical (grayscale content): {int(img_info.rgb_but_gray.sum())}")
print(f"Width {img_info.width.min()}-{img_info.width.max()} px, height {img_info.height.min()}-{img_info.height.max()} px")
display(img_info.groupby("organ")[["width", "height"]].describe().round(0))
SUMMARY.update({
    "img_w_min": int(img_info.width.min()), "img_w_max": int(img_info.width.max()),
    "img_h_min": int(img_info.height.min()), "img_h_max": int(img_info.height.max()),
    "n_img_grayscale": int(img_info["mode"].isin(SINGLE_CHANNEL).sum()),
    "n_img_rgb_but_gray": int(img_info.rgb_but_gray.sum()),
})

# Figure E4: image sizes, one panel per organ
panel_organs = [o for o in organs if o in set(img_info.organ)]
fig, axes = plt.subplots(1, len(panel_organs), figsize=(3.6 * len(panel_organs), 3.4), sharex=True, sharey=True, squeeze=False)
for ax, o in zip(axes[0], panel_organs):
    d = img_info[img_info.organ == o]
    ax.scatter(d.width, d.height, s=22, color=C_TRAIN, edgecolor=SURFACE, linewidth=0.8)
    ax.set_title(f"{o} (n={len(d)})")
    ax.set_xlabel("Width (px)")
axes[0][0].set_ylabel("Height (px)")
fig.suptitle("Figure E4. Image sizes by organ", fontweight="semibold", color=INK)
fig.tight_layout()
save(fig, "fig_e4_image_sizes.png")

In [ ]:
# Figure E5: two sample images per organ, each with one question (closed PRES/ABN preferred)
rng = random.Random(SEED)
samples = []
for o in panel_organs:
    names = sorted(img_info.loc[img_info.organ == o, "image_name"])
    for name in rng.sample(names, min(2, len(names))):
        qs = df[df.image_name == name]
        pref = qs[(qs.answer_type == "CLOSED") & qs["qtypes"].map(is_presence)]
        row = (pref if len(pref) else qs).iloc[0]
        samples.append((name, o, row.question, row.answer))

ncols = 3
nrows = math.ceil(len(samples) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 5.0 * nrows), squeeze=False, layout="constrained")
for ax in axes.flat:
    ax.axis("off")
for ax, (name, o, q, a) in zip(axes.flat, samples):
    with Image.open(IMG_INDEX[name]) as im:
        ax.imshow(im.convert("L"), cmap="gray")
    ax.set_title(f"{o} · {name}\n" + textwrap.fill(f"Q: {q}", 38) + "\n" + textwrap.fill(f"A: {a}", 38),
                 fontsize=9, fontweight="normal", loc="left")
fig.suptitle("Figure E5. Sample images with a question and reference answer", fontweight="semibold", color=INK)
save(fig, "fig_e5_samples.png")
print("Look at Figure E5 and note crops, borders, text or arrows on the images for the doc.")

## 10. Splits for training and evaluation

* **Validation split.** About 10% of training questions, taken only from *training-only* images (images with no test questions). Removing them from training therefore leaves every test image's exposure unchanged, and validation questions sit on images the model never trains on.
* **Image-disjoint split.** All images split 80/20 by image, for the secondary memorization check.

In [ ]:
train_only_imgs = sorted(img_train - img_test)
rng = random.Random(SEED)
rng.shuffle(train_only_imgs)
q_per_train_img = df[df.split == "train"].groupby("image_name").size()
target = 0.10 * n_train
val_imgs, count = [], 0
for im_name in train_only_imgs:
    if count >= target:
        break
    val_imgs.append(im_name)
    count += int(q_per_train_img[im_name])
if count < target:
    print(f"WARNING: training-only images hold only {count} questions (< 10% of train).")
assert not (set(val_imgs) & img_test)

df["split_with_val"] = df["split"]
df.loc[(df.split == "train") & df.image_name.isin(val_imgs), "split_with_val"] = "val"
print("Official split with validation:", df.split_with_val.value_counts().to_dict(),
      f"| validation images: {len(val_imgs)}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
_, te_idx = next(gss.split(df, groups=df.image_name))
df["split_image_disjoint"] = "train"
df.loc[df.index[te_idx], "split_image_disjoint"] = "test"
dj_tr = set(df.loc[df.split_image_disjoint == "train", "image_name"])
dj_te = set(df.loc[df.split_image_disjoint == "test", "image_name"])
assert not (dj_tr & dj_te), "image-disjoint split leaks images"
print("Image-disjoint split:", df.split_image_disjoint.value_counts().to_dict(),
      f"| images train {len(dj_tr)}, test {len(dj_te)}")

df[["qid", "image_name", "question", "answer", "answer_type", "question_type",
    "split", "split_with_val", "split_image_disjoint"]].to_csv(OUT_DIR / "splits.csv", index=False)
SUMMARY.update({
    "n_val_images": len(val_imgs), "n_val_q": int((df.split_with_val == "val").sum()),
    "n_disjoint_train_q": int((df.split_image_disjoint == "train").sum()),
    "n_disjoint_test_q": int((df.split_image_disjoint == "test").sum()),
    "n_disjoint_test_images": len(dj_te),
})

## 11. Polarity-labeling sheet

Two team members label each risk-subset question independently, in `label_A` and `label_B`, with one of:

| Label | Meaning | Example |
| --- | --- | --- |
| `present_if_yes` | "yes" means the finding is present | "Is there a pneumothorax?" |
| `absent_if_yes` | "yes" means the finding is absent | "Are the lungs clear?" / "Is the image normal?" |
| `not_a_finding` | Asks about normal anatomy or technique, not a finding | "Are the kidneys visible?" |
| `unclear` | Cannot decide from the question alone | — |

A third member fills `adjudicated` where A and B disagree. Save the filled sheet as `eda_outputs/polarity_labels_filled.csv`, then rerun the next two cells.

In [ ]:
pol = risk_subset[["qid", "image_name", "image_organ", "question_type", "question", "answer"]].copy()
for c in ["label_A", "label_B", "adjudicated", "notes"]:
    pol[c] = ""
pol.to_csv(OUT_DIR / "polarity_labels_template.csv", index=False)
print(f"Wrote {len(pol)} questions to {OUT_DIR / 'polarity_labels_template.csv'}")

In [ ]:
FILLED = OUT_DIR / "polarity_labels_filled.csv"
if FILLED.exists():
    from sklearn.metrics import cohen_kappa_score
    lab = pd.read_csv(FILLED, dtype=str).fillna("")
    both = lab[(lab.label_A != "") & (lab.label_B != "")]
    kappa = cohen_kappa_score(both.label_A, both.label_B)
    print(f"Cohen's kappa over {len(both)} questions: {kappa:.3f}")
    display(both[both.label_A != both.label_B][["question", "answer", "label_A", "label_B", "adjudicated"]])
    SUMMARY["polarity_kappa"] = round(kappa, 3)
else:
    print("Not labeled yet: fill polarity_labels_template.csv and save it as polarity_labels_filled.csv.")

## 12. Values for the Phase 3 doc

In [ ]:
DOC_KEYS = [
    "pct_closed", "top_qtype", "n_top_qtype", "q_per_image_median", "q_per_image_max",
    "closed_train_n", "closed_train_yes_pct", "closed_train_no_pct", "closed_train_other_pct",
    "closed_test_n", "closed_test_yes_pct", "closed_test_no_pct", "closed_test_other_pct",
    "presence_test_n", "presence_test_yes_pct", "presence_test_no_pct", "majority_closed_test_acc",
    "n_closed_nonyesno", "n_variant_groups", "n_case_only_groups", "n_open_unique", "n_open_singleton", "open_answer_words_median",
    "n_missing_question_type", "n_missing_answer_type", "n_missing_image_organ", "n_missing_image_file",
    "n_img_shared", "n_img_test", "n_test_para_in_train", "n_test_neardup", "n_exact_dup_cross",
    "img_w_min", "img_w_max", "img_h_min", "img_h_max", "n_img_grayscale",
    "n_test_abn_open", "n_val_images", "n_val_q", "n_disjoint_train_q", "n_disjoint_test_q",
]
print("Doc placeholders:")
for k in DOC_KEYS:
    print(f"  [{k}] = {SUMMARY.get(k, 'not computed')}")
print("\nOther checks:")
for k in sorted(set(SUMMARY) - set(DOC_KEYS)):
    print(f"  {k} = {SUMMARY[k]}")
with open(OUT_DIR / "eda_summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print(f"\nSaved {OUT_DIR / 'eda_summary.json'}; figures in {FIG_DIR}/")